# 6장 — 늦게 오는 소식: 지연과 출렁임 (핵심)

웹 챕터(6장)와 **같은 모형, 같은 수치**를 파이썬으로 계산합니다:
샤워기(균형 루프 + 배관 지연)의 과잉 교정, 출렁임이 커지는 경계, 그리고 EMA = 1차 정보 지연.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch6')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 샤워기: 균형 루프 + 지연

손잡이가 정하는 온도 V, 머리에서 느끼는 온도는 배관을 지나는 D초 전의 값 $H(t) = V(t - D)$.
사람은 느끼는 온도를 보고 손잡이를 돌립니다:

$$\frac{dV}{dt} = \frac{38 - H(t)}{\tau}$$

처음 물 20°C, 손잡이 범위는 찬물 10°C ~ 뜨거운 물 60°C. 0.01초 간격의 오일러법으로 60초를 계산합니다(웹 sim.ts showerRun 과 같은 계산).

In [ ]:
G, V0, T, dt, LO, HI = 38, 20, 60, 0.01, 10, 60

def shower(D, tau):
    n, lag = round(T / dt), round(D / dt)
    buf = [V0] * (lag + 1)      # D초 동안 배관 속에 있는 물(순환 버퍼)
    idx, V, H = 0, V0, []
    for k in range(n + 1):
        h = buf[idx] if lag > 0 else V
        H.append(h)
        V = min(HI, max(LO, V + (G - h) / tau * dt))
        if lag > 0:
            buf[idx] = V
            idx = (idx + 1) % (lag + 1)
    return H

def summary(D, tau):
    H = shower(D, tau)
    peak = max(H)
    late = max(abs(h - G) for h in H[-1000:])      # 마지막 10초의 출렁임
    return peak, H.index(peak) * dt, late

print('  D    τ   D/τ    최고 온도      마지막 10초 출렁임')
for D, tau in [(0.5, 2), (1, 2), (2, 2), (3, 2), (3.5, 2), (2, 3), (2, 4)]:
    peak, tp, late = summary(D, tau)
    print(f'{D:4.1f} {tau:4.1f}  {D / tau:4.2f}   {peak:6.2f}°C ({tp:4.1f}초)   ±{late:5.2f}°C')

# 웹 실험실(DelayLab)·본문 수치 자가검증
peak, tp, late = summary(2, 2)
assert round(peak, 2) == 47.05 and abs(tp - 6.0) < 0.01 and late < 0.1   # 기본값: 최고 47.05°C, 가라앉음
assert summary(0.5, 2)[0] <= G + 0.01                                     # D/τ = 0.25: 넘치지 않음
assert 38.5 < summary(1, 2)[0] < 39                                       # 0.5: 살짝 넘침
assert summary(3, 2)[2] > 10                                              # 1.5: 60초 뒤에도 크게 출렁임
assert summary(3.5, 2)[0] == HI                                           # 1.75: 끝까지 돌림(60°C)
print('✅ 웹과 일치: D = τ = 2초면 최고 47.05°C · D/τ 가 클수록 출렁임이 커진다')

## 2. 출렁임이 커지기 시작하는 선

샤워기 식은 **지연 미분방정식**입니다. 알려진 경계: $D/\tau < 1/e \approx 0.37$ 이면 넘치지 않고,
$1/e$ 와 $\pi/2 \approx 1.57$ 사이면 출렁이며 가라앉고, $\pi/2$ 를 넘으면 출렁임이 커집니다.
손잡이 범위를 풀어 두고(제한 없음) 출렁임의 크기가 줄어드는지 커지는지 봅시다.

In [ ]:
import math

def growth(D, tau, T_=120):
    # 손잡이 범위 제한 없이: 30~60초 출렁임 크기 vs 90~120초 출렁임 크기
    n, lag = round(T_ / dt), round(D / dt)
    buf = [V0] * (lag + 1); idx, V, H = 0, V0, []
    for k in range(n + 1):
        h = buf[idx]
        H.append(h)
        V = V + (G - h) / tau * dt
        buf[idx] = V; idx = (idx + 1) % (lag + 1)
    a1 = max(abs(h - G) for h in H[3000:6000])
    a2 = max(abs(h - G) for h in H[9000:12000])
    return a2 / a1

print(f'1/e = {1 / math.e:.3f},  π/2 = {math.pi / 2:.3f}')
for r in [1.0, 1.4, 1.5, 1.6, 1.7]:
    g = growth(2 * r, 2)
    print(f'D/τ = {r:.2f}: 60초 뒤 출렁임 크기 비율 {g:7.3f}  →  {"줄어듦" if g < 1 else "커짐"}')
assert growth(3.0, 2) < 1 < growth(3.2, 2)        # 경계는 1.5 와 1.6 사이 (π/2 ≈ 1.571)
print('✅ 출렁임이 줄다가 커지기 시작하는 경계 ≈ π/2')

## 3. EMA = 1차 정보 지연

$$\frac{d\,\text{보이는 값}}{dt} = \frac{\text{실제 값} - \text{보이는 값}}{\tau} \;\longrightarrow\; m_k = \beta m_{k-1} + (1-\beta) g_k,\quad \beta = 1 - \frac{DT}{\tau}$$

입력: 20번째 걸음부터 방향이 0 → 1로 바뀌고, 매 걸음 ±0.3 지그재그로 흔들리는 '기울기' 신호(웹 emaInput 과 같음).

In [ ]:
N = 100

def ema_input(noise=True, step=True):
    return [(1 if step and k >= 20 else 0) + (0.3 * (-1) ** k if noise else 0) for k in range(N)]

def ema(beta, g):
    m, out = 0.0, []
    for x in g:
        m = beta * m + (1 - beta) * x
        out.append(m)
    return out

for beta in [0.5, 0.8, 0.9, 0.95, 0.99]:
    step = ema(beta, ema_input(False))
    zig = ema(beta, ema_input(True, False))                 # 흔들림만 따로 (지그재그 폭 0.6)
    wiggle = max(zig[80:]) - min(zig[80:])
    print(f'β = {beta:4.2f}: 평균 지연 {1 / (1 - beta):5.0f}걸음,  10걸음 뒤 따라온 정도 {step[29] * 100:5.1f}%,  남은 흔들림 폭 {wiggle:.4f} (이론 {0.6 * (1 - beta) / (1 + beta):.4f})')
    assert abs(wiggle - 0.6 * (1 - beta) / (1 + beta)) < 0.002

assert abs(ema(0.9, ema_input(False))[29] - (1 - 0.9 ** 10)) < 1e-12      # 약 65%
assert round(1 / (1 - 0.9)) == 10 and round(1 / (1 - 0.999)) == 1000      # Adam β₁·β₂
# 계산 연습: β = 0.9, 1 이 두 번
m1 = 0.9 * 0 + 0.1 * 1
m2 = 0.9 * m1 + 0.1 * 1
assert abs(m1 - 0.1) < 1e-12 and abs(m2 - 0.19) < 1e-12
# SMOOTH(1차 지연)를 오일러법(DT = 1)으로 계산하면 EMA 와 같다
tau = 10
smooth, s = [], 0.0
for x in ema_input():
    s = s + (x - s) / tau          # 보이는 값 += (실제 − 보이는 값) × DT/τ
    smooth.append(s)
assert all(abs(a - b) < 1e-12 for a, b in zip(smooth, ema(1 - 1 / tau, ema_input())))
print('✅ EMA = 1차 정보 지연(β = 1 − DT/τ) · β = 0.9 → 10걸음 뒤 65% · 흔들림 ↔ 반응 속도의 맞바꿈')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
ts = [k * dt for k in range(round(T / dt) + 1)]
for (D, tau), c in zip([(0.5, 2), (2, 2), (3, 2)], ['#10b981', '#f59e0b', '#e11d48']):
    ax1.plot(ts, shower(D, tau), color=c, lw=2, label=f'D = {D}초, τ = {tau}초 (D/τ = {D / tau:.2f})')
ax1.axhline(G, ls='--', color='#2563eb')
ax1.set_xlabel('초'); ax1.set_ylabel('°C'); ax1.set_title('샤워기: 지연이 길수록 크게 출렁인다'); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
g = ema_input()
ax2.plot(range(N), g, color='#94a3b8', lw=1, label='입력(기울기)')
for beta, c in [(0.5, '#c4b5fd'), (0.9, '#7c3aed'), (0.99, '#4c1d95')]:
    ax2.plot(range(N), ema(beta, g), color=c, lw=2, label=f'EMA β = {beta}')
ax2.set_xlabel('걸음'); ax2.set_title('EMA: β가 클수록 덜 흔들리고 늦게 따라온다'); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 오늘의 발견

- **균형 루프 + 지연 = 과잉 교정과 출렁임.** D = τ = 2초인 샤워기는 목표 38°C인데 최고 47°C.
- 출렁임의 세기는 **D ÷ τ**가 정한다: 0.37 아래면 넘치지 않고, π/2 ≈ 1.57을 넘으면 출렁임이 커진다.
- 처방: 덜 조급하게(τ↑) 또는 빨리 보기(D↓).
- **1차 정보 지연 = 지수 평활 = EMA**(β = 1 − DT/τ, 평균 지연 1/(1 − β) 걸음). Adam β₁ = 0.9 → 10걸음, β₂ = 0.999 → 1,000걸음.
- EMA는 흔들림을 줄이는 대신 늦게 반응한다. 이 맞바꿈이 β를 고르는 이유.

다음 장(7장 커지다 멈추는 성장)에서 강화 루프와 균형 루프가 만나는 S자 곡선을 봅니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**